In [8]:
import pandas as pd
#from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from subprocess import CREATE_NO_WINDOW
from webdriver_manager.core.os_manager import OperationSystemManager,ChromeType

In [3]:
url = 'https://stock.naver.com/domestic/stock/069500/info/summary'

In [ ]:
br_ver = OperationSystemManager().get_browser_version_from_os(ChromeType.GOOGLE)
version_main=int(br_ver.split('.')[0])
'Dirver Setting'

option = Options()
option.add_argument('--disable-gpu')
option.add_argument('--window-size=1920x1080')
option.add_argument('--start-maximized')
option.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36')
service = Service()
service.creation_flags = CREATE_NO_WINDOW

# driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver = uc.Chrome(service=service, options=option, version_main=version_main)
driver.implicitly_wait(3) # 화명 렌더링을 3초간 기다림
str1 = driver.capabilities['browserVersion']
str2 = driver.capabilities['chrome']['chromedriverVersion'].split(' ')[0]

print(f'chrome browser version : {str1[0:2]}, chrome dirver version : {str2[0:2]}')

driver.get(url)
# divs = driver.find_elements(By.CLASS_NAME, "StockInfo_listing-info__qzcRk")

li_elements = driver.find_elements(By.XPATH, '//ul[@class="StockInfo_listing-info__qzcRk"]/li') # //ul[@class="StockInfo_listing-info__qzcRk"]/li

results = []

for li in li_elements:
    # print(li)
    spans = li.find_elements(By.TAG_NAME, "span")
    data = [span.text for span in spans if span.text.strip() != ""]
    if data:
        item = {
            "지표": data[0],
            "값": data[1] if len(data) > 1 else 'N/A'
        }
        results.append(item) # 리스트에 추가
        # print(f"지표: {item['지표']}, 값: {item['값']}") # 기존 출력 유지


# 1. 특정 캡션(포트폴리오 구성)을 포함한 테이블을 먼저 찾습니다.
# "069500 포트폴리오 구성" 텍스트가 포함된 caption의 부모 table을 타겟팅합니다.
# target_table = driver.find_element(By.XPATH, "//table[contains(caption, '포트폴리오 구성')]")
# target_table_xpath = "//table[contains(caption, '포트폴리오 구성')]/following::table[@class='InnerTable_table___xmXR'][1]"
# target_table = driver.find_element(By.XPATH, target_table_xpath)

all_tables = driver.find_elements(By.CSS_SELECTOR, "table.InnerTable_table___xmXR")
seen_names = set()  
if len(all_tables) >= 2:
    target_table = all_tables[0] 
    rows = target_table.find_elements(By.CSS_SELECTOR, "tbody.InnerTable_tbody__zuUyv tr")
    
    row_num = 0
    for row in rows:
        if row_num <= 10:
            cells = row.find_elements(By.TAG_NAME, "td")
            print([cell.text for cell in cells])
            # raw_data = [cell.text for cell in cells]
            row_data = [cell.text.replace('\n', ' ').strip() for cell in cells]
            stock_name = row_data[0]
            if stock_name not in seen_names:

                if len(row_data) > 3:
                    continue
                if not any('10년' in str(cell) for cell in row_data):
                    continue

                if not row_data or '10년' in row_data[0]:
                    continue

                item = {
                    "종목명": row_data[0],
                    "주식수": row_data[1],
                    "비중중": row_data[2],
                    "시세": row_data[3] if len(row_data) > 3 else 'N/A',
                    "전일대비": row_data[4] if len(row_data) > 4 else 'N/A'
                    # 필요에 따라 더 추가 가능
                }
                table_results.append(item)
                seen_names.add(stock_name)
                print(row_data)

            row_num += 1
            print(row_num)




# /        if row_data:
            # 데이터가 [종목명, 비중, 가격...] 순서라고 가정할 때 ['삼성전기', '84\n주', '1.32\n%', '901,000', '하락\n13,000\n(-1.42%)']


print(table_results)
driver.quit()

chrome browser version : 14, chrome dirver version : 14
['삼성전자', '7,022\n주', '32.56\n%', '285,500', '상승\n17,000\n(+6.33%)']
['SK하이닉스', '834\n주', '24.28\n%', '1,880,000', '상승\n194,000\n(+11.51%)']
['SK스퀘어', '139\n주', '2.63\n%', '1,187,000', '상승\n89,000\n(+8.11%)']
['현대차', '205\n주', '2.17\n%', '646,000', '상승\n33,000\n(+5.38%)']
['두산에너빌리티', '653\n주', '1.46\n%', '128,000', '하락\n1,600\n(-1.23%)']
['KB금융', '482\n주', '1.34\n%', '158,800', '하락\n2,900\n(-1.79%)']
['삼성전기', '84\n주', '1.32\n%', '900,000', '하락\n14,000\n(-1.53%)']
['한화에어로스페이스', '49\n주', '1.10\n%', '1,315,000', '상승\n8,000\n(+0.61%)']
['삼성SDI', '92\n주', '1.07\n%', '684,000', '상승\n6,000\n(+0.88%)']
['삼성물산', '145\n주', '1.05\n%', '452,000', '상승\n29,500\n(+6.98%)']
[{'종목명': '10년', '주식수': '19.27%', '비중중': '19.19%', '시세': 'N/A', '전일대비': 'N/A'}, {'종목명': '10년', '주식수': '19.27%', '비중중': '19.19%', '시세': 'N/A', '전일대비': 'N/A'}, {'종목명': '10년', '주식수': '19.27%', '비중중': '19.19%', '시세': 'N/A', '전일대비': 'N/A'}, {'종목명': '10년', '주식수': '19.27%', '비중중': '19.

In [27]:
results

[{'지표': '기초지수\n(추적지수)', '값': '코스피 200'},
 {'지표': '상장일', '값': '2002. 10. 14.'},
 {'지표': '운용사', '값': '삼성자산운용(주)'},
 {'지표': '시가총액', '값': '26조 1,322억'},
 {'지표': '운용자산', '값': '24조 8,758억'},
 {'지표': '레버리지', '값': '1배'},
 {'지표': 'NAV', '값': '115,782'},
 {'지표': '괴리율', '값': '0.02%'},
 {'지표': '총보수', '값': '0.15%'},
 {'지표': '추적오차율', '값': '0.41%'},
 {'지표': '증권거래', '값': '-'},
 {'지표': '매매차익', '값': '비과세'},
 {'지표': '배당소득세', '값': '-'}]